In [1]:
import cobra
from corda import CORDA

import json

import pandas as pd
import random

In [15]:
# load inputs
build_files_path = '/data2/hratch/human_me/build_files/'
full_model = cobra.io.read_sbml_model('/data2/hratch/human_me/input_files/recon2_2.xml')

rmd = json.load(open(build_files_path + "required_metabolic_model_metabolites.json"))
rmd = [v for k,v in rmd.items() if k == 'c']
rmd = [item for sublist in rmd for item in sublist]

In [3]:
def build_toy_model(full_model, rmd, max_score = 3, frac_reactions = 0.01):
    '''Generate small toy model from full Recon2.2'''

    # generate reaction confidence scores
    n_reactions = len(full_model.reactions)
    n_reactions_to_keep = round(frac_reactions*n_reactions)

    one_reaction = [r.id for r in full_model.reactions if len(r.genes)==1]
    reactions_to_exclude = random.sample(one_reaction, round(len(one_reaction)*.25)) # decreases prob of choosing a reaction with a gene by 4x
    reactions_to_exclude += [r.id for r in full_model.reactions if len(r.genes)>1]


    population = sorted(set([r.id for r in full_model.reactions]).difference(reactions_to_exclude))
    reactions_to_keep = random.sample(population, k = n_reactions_to_keep)

    conf = {}
    for r in full_model.reactions: 
        if r.id not in reactions_to_keep:
            conf[r.id] = -1
        else:
            conf[r.id] = random.choice(list(range(max_score +1)))
    conf["biomass_reaction"] = 3
    
    print('Extract model')
    opt = CORDA(full_model, conf, met_prod = rmd)
    opt.build()
    print(opt)
    
    print('Generate cobra model')
    toy_model = opt.cobra_model('toy_model')
    
    return toy_model

In [ ]:
iter_, max_iter = 0, 10
first = True
min_growth = 1e-3
toy_model = cobra.Model('')
opt_val = toy_model.slim_optimize()

while (iter_ < max_iter) and opt_val < min_growth:
    print('iteration: {}'.format(iter_))
    
    toy_model = build_toy_model(full_model, rmd)
    opt_val = toy_model.slim_optimize()

    print('Growth value: {}'.format(opt_val))
    print('--------')
    iter_ += 1

print(opt_val)
if opt_val >= min_growth:
    cobra.io.write_sbml_model(cobra_model = toy_model, 
                              filename = '/data2/hratch/human_me/input_files/toy_model.xml')

iteration: 0
Extract model


In [12]:
minimal_model = build_toy_model(full_model, rmd, max_score = 3, frac_reactions = 0)

Extract model
build status: reconstruction complete
Inc. reactions: 519/8555
 - unclear: 0/0
 - exclude: 471/8507
 - low and medium: 0/0
 - high: 48/48

Generate cobra model


In [ ]:
minimal_model